# mine-tuning-model

## 필수 라이브러리 설치

In [ ]:
!pip install --upgrade chromadb langchain langchain-community langchain-openai sentence-transformers transformers datasets gradio langgraph -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 150.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip freeze > /content/drive/MyDrive/Colab Notebooks/Github/mine-tuning-model/requirements.txt

## 허깅페이스에서 데이터 가져오기

In [ ]:
from datasets import load_dataset

ds = load_dataset("naklecha/minecraft-question-answer-700k")
print(ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/943 [00:00<?, ?B/s]

train.json:   0%|          | 0.00/314M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/694814 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['answer', 'question', 'source'],
        num_rows: 694814
    })
})


# 데이터 임베딩


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("BAAI/bge-large-en-v1.5")

print('[입베딩 성공]')
print(f"임베딩 차원: {embedding_model.get_sentence_embedding_dimension()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

[입베딩 성공]
임베딩 차원: 1024


/tmp/ipykernel_3452/795011968.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"임베딩 차원: {embedding_model.get_sentence_embedding_dimension()}")


## Chroma DB 구축

In [ ]:
import chromadb
from tqdm import tqdm

COLLECTION_NAME = "minecraft_rag"
CHROMA_DIR      = "/content/chroma_db"

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
existing      = [c.name for c in chroma_client.list_collections()]

# 기존 컬렉션 있고 데이터도 있으면 재사용
if COLLECTION_NAME in existing:
    collection = chroma_client.get_collection(name=COLLECTION_NAME)
    if collection.count() > 0:
        print(f"✅ 기존 chroma_db 로드 완료: {collection.count()}개 데이터")
    else:
        # 데이터 0개면 삭제 후 새로 구축
        print("⚠️ 데이터 0개 — 삭제 후 새로 구축합니다...")
        chroma_client.delete_collection(name=COLLECTION_NAME)
        existing = []  # 아래 else 블록 타도록 초기화

if COLLECTION_NAME not in existing:
    # 🆕 새로 구축
    print("[ChromaDB 새로 구축 중...]")
    collection = chroma_client.create_collection(name=COLLECTION_NAME)

    total_size = len(ds["train"]["answer"])
    answers    = ds["train"]["answer"]
    questions  = ds["train"]["question"]
    sources    = ds["train"]["source"]
    print(f"총 데이터 수: {total_size}개")

    # ============================================================
    # [테스트용 5000개 - 주석 처리]
    # sample_size = 5000
    # batch_size  = 500
    # for i in range(0, sample_size, batch_size):
    #     batch_answers = answers[i:i+batch_size]
    #     embeddings = embedding_model.encode(batch_answers, show_progress_bar=True).tolist()
    #     collection.add(
    #         documents=batch_answers,
    #         embeddings=embeddings,
    #         metadatas=[{"question": q, "source": s} for q, s in zip(questions[i:i+batch_size], sources[i:i+batch_size])],
    #         ids=[str(j) for j in range(i, i+len(batch_answers))]
    #     )
    #     print(f" {min(i+batch_size, sample_size)}/{sample_size} 완료")
    # ============================================================

    # 전체 데이터 구축
    batch_size = 500
    for i in tqdm(range(0, total_size, batch_size), desc="벡터 DB 구축 중"):
        batch_answers   = answers[i:i+batch_size]
        batch_questions = questions[i:i+batch_size]
        batch_sources   = sources[i:i+batch_size]

        embeddings = embedding_model.encode(
            batch_answers,
            show_progress_bar=False
        ).tolist()

        collection.add(
            documents=batch_answers,
            embeddings=embeddings,
            metadatas=[{"question": q, "source": s} for q, s in zip(batch_questions, batch_sources)],
            ids=[str(j) for j in range(i, i+len(batch_answers))]
        )

    print(f"\n🎉 완료! 총 {collection.count()}개 저장됨")

[ChromaDB 새로 구축 중...]
총 데이터 수: 694814개


벡터 DB 구축 중:   3%|▎         | 48/1390 [01:35<44:30,  1.99s/it]


KeyboardInterrupt: 

## 검색 테스트

In [ ]:
def retrieve(query: str, top_k: int = 3):
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    return results["documents"][0], results["metadatas"][0]

# 테스트
docs, metas = retrieve("How to find diamonds?")
for i, (doc, meta) in enumerate(zip(docs, metas)):
    print(f"=== 검색 결과 {i+1} ===")
    print(f"source     : {meta['source']}")
    print(f"Q       : {meta['question']}")
    print(f"Answer  : {doc[:200]}")
    print()

## LLM 로드

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_id = "Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False
)

print("LLM 로드 완료")

## RAG 파이프라인 완성

In [ ]:
def rag_answer(query: str) -> str:
    # 1. 관련 문서 검색
    docs, metas = retrieve(query, top_k=3)
    context = "\n\n".join(docs)

    # 2. 프롬프트 구성
    prompt = f"""You are a helpful Minecraft expert assistant.
Use the following context to answer the question accurately.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {query}
Answer:"""

    # 3. LLM 답변 생성
    messages = [{"role": "user", "content": prompt}]
    result = llm(messages)
    answer = result[0]["generated_text"][-1]["content"]

    # 4. 출처 출력
    print(f"출처: {metas[0]['source']}\n")
    return answer


In [ ]:
# 테스트
response = rag_answer("How to make a Nether portal?")
print(response)

## Gradio 연결

In [ ]:
import gradio as gr

def chat(message, history):
    answer = rag_answer(message)
    return answer

demo = gr.ChatInterface(
    fn=chat,
    title="⛏️ Minecraft Guide LLM",
    description="Minecraft Wiki 기반 AI 가이드에게 무엇이든 물어보세요!",
    examples=[
        "How to find diamonds?",
        "How do I defeat the Ender Dragon?",
        "How to make a Nether portal?"
    ],
)

demo.launch(share=True)  # share=True 로 외부 접속 링크 생성